# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
import re

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/10/31 06:51:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/10/31 06:51:15 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
23/10/31 06:51:23 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


In [5]:
year0 = "2019"

In [6]:
year1 = str(int(year0) + 1)
#year1 = year0

# Set Parameters

Set date range and airports here.

In [7]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 + "-01-01"}

In [8]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

In [24]:
## get the route expansion and join with arincfix to get the icao_region for each fix

# df_plannedroute = (
#     api.dataframe("PlannedRoute", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
#     .select(
#         "primary_key",
#         "track_key",
#         "route",
#         "labels.is_lfpd",
#         F.col("total_route_length").alias("route_length_nm"),
#     )
#     .filter(F.col("track_key") == '49nIDzozfLLAAUE')
# #    .filter(F.col("labels.is_lfpd") == 'true')
#     .withColumn("route_length_nm", F.round("route_length_nm", 1))
# )

Multiple versions found: 3.1.24, 3.1.34, 3.1.40


In [25]:
# df_plannedroute.show(truncate=False)

+---------------+---------------+---------------------------------------------------------------------------------------------------------------------------------+-------+---------------+
|primary_key    |track_key      |route                                                                                                                            |is_lfpd|route_length_nm|
+---------------+---------------+---------------------------------------------------------------------------------------------------------------------------------+-------+---------------+
|4mQtKlmB-qABHOw|49nIDzozfLLAAUE|KBOS.BLZZR4.BLZZR..BAF.Q480.AIR.J110.EMPTY..FFO..RID..TILMN.J110.STL..HLV..MCI..HGO..HBU..MLF..ILC..RUMPS..OAL..INYOE.DYAMD5.KSFO|false  |2478.9         |
|4mQtKlmB-qABIQm|49nIDzozfLLAAUE|KBOS.HYLND5.HYLND..HANAA.Q816.ARNII..UTR..YSO..ASP..GRB..GEP.J70.ABR.J32.LLC..LEGGS.BDEGA3.KSFO/0549                             |false  |11882.2        |
|4mQtKlmB-qABIQn|49nIDzozfLLAAUE|KBOS.HYLND5.HYLND..HANAA.Q8

In [19]:
#airports = [ "KADW", ]

# UDFs

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [20]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

Define the schema of the litetrack points structure array

In [21]:
points_schema = T.StructType([
    T.StructField("points", T.ArrayType(
        T.StructType([
            T.StructField("primary_key", T.StringType(), True),
            T.StructField("time", T.LongType(), True),
            T.StructField("latitude", T.DoubleType(), True),
            T.StructField("longitude", T.DoubleType(), True),
            T.StructField("altitude", T.FloatType(), True),
            T.StructField("course", T.FloatType(), True),
            T.StructField("speed", T.FloatType(), True),
            T.StructField("source_point_keys", T.ArrayType(
                T.StructType([
                    T.StructField("element", T.StringType(), True),
                ]), True), True),
            T.StructField("acceleration", T.DoubleType(),True),
            T.StructField("type", T.StringType(), True),
            T.StructField("airspace_key", T.StringType(), True),
            T.StructField("tas", T.FloatType(), True),
            T.StructField("ias", T.FloatType(), True),
            T.StructField("along_track_distance", T.FloatType(), True),
            T.StructField("derived_point_key", T.StringType(), True),
        ]), True), True)
    ])

Define a function that will convert the points array into a string containing n-tuples of data for each litetrack point

In [22]:
none_string = "*"
separator_string = ":"

def stringify_points(array_of_points:T.ArrayType(points_schema)) -> str:
    output = ""
    for x in array_of_points:
        output += (none_string if x.time is None else str(x.time)) + " "
        output += (none_string if x.latitude is None else f'{x.latitude:.6f}') + " "
        output += (none_string if x.longitude is None else f'{x.longitude:.6f}') + " "
        output += (none_string if x.altitude is None else f'{x.altitude:.0f}') + " "
        output += (none_string if x.course is None else f'{x.course:.0f}') + " "
        output += (none_string if x.speed is None else f'{x.speed:.0f}') + separator_string

    # drop the trailing tuple separator
    if len(array_of_points) > 0:
        output = output[0:len(separator_string) * -1]
    return output

#stringify_points_udf = F.udf(stringify_points, T.StringType())

In [23]:
def extract_carrier(callsign) -> str:
    regex = "^([A-Za-z]+)\d*[A-Za-z]*$"
    match = re.search(regex, callsign)
    carrier = ""
    if match:
        carrier = match.group(1)
    return carrier

extract_carrier_udf = F.udf(extract_carrier, T.StringType())

In [24]:
#print(extract_carrier("aal2343"))

# Load LiteTrack

the points array is "stringified" to a new `trackpoints` column

In [25]:
#df_lt_meta = api.dataframe(
#    "LiteTrack", **dates, partition_filters=api.custom_partitions("ASSOCIATED")
#).select(
#    "track_key",
#    F.col("start_time").alias("start_epoch_millisec"),
#    F.col("end_time").alias("end_epoch_millisec"),
#)

In [26]:
df_litetrack = (
    api.dataframe(
        "LiteTrack", **dates, partition_filters=api.custom_partitions("ASSOCIATED")
    )
#    .withColumn("trackpoints", stringify_points_udf("points"))
    .select(
        "track_key",
        F.col("start_time").alias("start_epoch_millisec"),
        F.col("end_time").alias("end_epoch_millisec"),
#        "mode_s_code",
#        "trackpoints",
#        "points",
    )
)

Could not find data for: 2019-01-30, 2019-02-14
Multiple versions found: 3.1.27, 3.1.30


In [27]:
#df_litetrack.count()

# Create route expansion from ArincFix and PlannedRoute

In [28]:
## get fix_key and associated icao_region

df_arincfix = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .select(
        F.col("primary_key").alias("fix_key"),
        F.col("identification.icao_region").alias("icao_region"),
    )
)

Multiple versions found: 3.1.12.2, 3.1.22


In [29]:
## get the route expansion and join with arincfix to get the icao_region for each fix

df_plannedroute = (
    api.dataframe("PlannedRoute", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
    .select(
        "track_key",
#        "file_time",
#        "departure_state",
#        "callsign",
#        "aircraft_type",
#        F.col("departure_aerodrome").alias("orig"),
#        F.col("destination_aerodrome").alias("dest"),
        "route",
        F.col("total_route_length").alias("route_length_nm"),
        F.col("labels.is_lfpd"),
        F.explode("legs").alias("leg"),
        F.col("leg.sequence_number").alias("seqnum"),
        F.col("leg.path_terminator.fix_name").alias("fix_name"),
        F.col("leg.path_terminator.fix_key").alias("fix_key"),
#        F.col("leg.path_terminator.latitude").alias("latitude"),
#        F.col("leg.path_terminator.longitude").alias("longitude"),
#        F.col("leg.path_terminator.magnetic_variation.modeled").alias("magnetic_variation"),
#        F.col("metadata.start_date").alias("start_date"),
#        F.col("metadata.end_date").alias("end_date"),
    )
#    .filter(F.col("track_key") == '41PpJCvdMXJABBt')
#    .filter(F.col("departure_state") == 'PRE')
    .filter(F.col("labels.is_lfpd") == 'true')
    .withColumn("route_length_nm", F.round("route_length_nm", 1))
    .join(df_arincfix, on=["fix_key"], how="inner")
    .withColumn("fix_name_uniq", F.concat("fix_name", F.lit("("), "icao_region", F.lit(")")))
    .drop("legs", "leg", "is_lfpd", "icao_region", "fix_key")
#    .orderBy("track_key", "seqnum")
)

Multiple versions found: 3.1.24, 3.1.34, 3.1.40


In [30]:
#df_plannedroute.printSchema()

In [31]:
#df_plannedroute.show()

In [32]:
#remove any duplicate fixes in the route expansion

window = Window.partitionBy("track_key", "fix_name").orderBy(col("seqnum").asc())
df_plannedroute_dedup = (
    df_plannedroute
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
    .orderBy("track_key", "seqnum")
)

In [33]:
#df_plannedroute_dedup.printSchema()

In [34]:
#df_plannedroute_dedup.show()

In [35]:
# make route expansion string as fix_name_expansion and fix_name_uniq_expansion 

df_route_expansion = (
    df_plannedroute_dedup
        .groupby("track_key", "route_length_nm", "route")
        .agg(F.sort_array(F.collect_list(F.struct("seqnum", "fix_name"))).alias("fix_name_list"), F.sort_array(F.collect_list(F.struct("seqnum", "fix_name_uniq"))).alias("fix_name_uniq_list"))
        .withColumn("fix_name_expansion", F.concat_ws(".", F.col("fix_name_list.fix_name")))
        .withColumn("fix_name_uniq_expansion", F.concat_ws(".", F.col("fix_name_uniq_list.fix_name_uniq")))
        .drop("fix_name_list", "fix_name_uniq_list")
)

In [36]:
#df_route_expansion.printSchema()

In [37]:
#df_route_expansion.show(truncate=False)

In [38]:
#df_route_expansion.count()

# Load Flightplan Series

Here, we'll load the FlightplanSeries and perform the following:

1. Filter flight plans so either the arrival or departure airport is for an airport we want
2. Ensure arrival and departure airports are not the same airport
3. Convert dates from unix time to YYYYMMDD using a previously defined UDF
4. Extract year and month from date (for partitioning later)
5. Join metadata from LiteTrack (only needed to add in start_time and end_time)
6. Join route expansion information

In [39]:
## get flightplans for the arrivals to and departures from airports_of_interest

df_fps0 = (
    api.dataframe("FlightplanSeries", **dates, metadata=True)
    .select(
        "track_key",
        to_date("metadata.effective_start_date").alias("start_date"),
        to_date("metadata.effective_end_date").alias("end_date"),
        "callsign",
        "aircraft_type",
        "mode_s_code",
        F.col("initial_departure_aerodrome").alias("orig"),
        F.col("final_destination_aerodrome").alias("dest"),
#        "ffpd_route",
#        "lfpd_route",
    )
    .withColumn("month", F.substring("start_date", 5, 2))
#    .withColumn("carrier", extract_carrier_udf("callsign"))
    .withColumn("carrier", F.regexp_extract("callsign", "^([A-Za-z]+)\d*[A-Za-z]*$", 1))
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
#    .filter((F.col("orig") == "KSHV") & (F.col("dest") == "KATL"))
#    .join(df_litetrack, on=["track_key"], how="inner")
#    .join(df_route_expansion, on=["track_key"], how="inner")
)

Multiple versions found: 3.1.20, 3.1.34


In [40]:
df_fps1 = df_fps0.join(df_route_expansion, on=["track_key"], how="inner")

In [41]:
df_fps = df_fps1.join(df_litetrack, on=["track_key"], how="inner")

In [42]:
#df_fps.show()

# Split out Arrivals and Departures

So we can save data into files pertaining to arrivals and departures, we'll:

1. Split data into arrivals and departures sets, adding an `operation column` to each and marking it as `ARRIVALS` or `DEPARTURES`
2. Re-apply filter to only airports we care about
3. Merge them back into one dataset afterwards

In [43]:
df_arrivals = (
    df_fps.withColumn("airport", F.col("dest"))
    .withColumn("operation", F.lit("ARRIVALS"))
    .filter(F.col("airport").isin(airports))
)

In [44]:
df_departures = (
    df_fps.withColumn("airport", F.col("orig"))
    .withColumn("operation", F.lit("DEPARTURES"))
    .filter(F.col("airport").isin(airports))
)

In [45]:
df_fps_combined = df_arrivals.unionByName(df_departures)

# Save FlightplanSeries Data

create the fps output dataframe

In [46]:
df_fps_output = (
    df_fps_combined
    .select(
        "track_key",
        "start_date",
        "end_date",
        "start_epoch_millisec",
        "end_epoch_millisec",
        "orig",
        "dest",
        "callsign",
        "carrier",
        "mode_s_code",
        "aircraft_type",
#        "ffpd_route",
#        "lfpd_route",
        "route",
        "fix_name_expansion",
        "fix_name_uniq_expansion",
        "route_length_nm",
        "airport",
        "operation",
        "month",
    )
)

In [47]:
df_fps_output.show()

+---------------+----------+--------+--------------------+------------------+----+----+--------+-------+-----------+-------------+--------------------+--------------------+-----------------------+---------------+-------+---------+-----+
|      track_key|start_date|end_date|start_epoch_millisec|end_epoch_millisec|orig|dest|callsign|carrier|mode_s_code|aircraft_type|               route|  fix_name_expansion|fix_name_uniq_expansion|route_length_nm|airport|operation|month|
+---------------+----------+--------+--------------------+------------------+----+----+--------+-------+-----------+-------------+--------------------+--------------------+-----------------------+---------------+-------+---------+-----+
|4-EtP_7X18AAA1b|  20190315|20190316|       1552647914039|     1552650387500|KLEB|KBOS|  KAP325|    KAP|     A69D00|         C402|KLEB..LEB.V141.CO...|LEB.KELLI.KERSY.C...|   LEB(K6).KELLI(K6)...|          103.4|   KBOS| ARRIVALS|   03|
|4-EtP_7X18AAADo|  20190315|20190316|       15526521

Here, we'll partition by a few fields:

`AIRPORT / MONTH / OPERATION TYPE`

the .repartition(cols) "...shuffles the data around between all the processors, so that all data specific to a combination of airport/operation/year/month goes to one processor" and then produces a single file

In [63]:
(
    df_fps_output
    .repartition("airport", "operation", "month")
    .write.option("header", True).partitionBy(["airport", "operation", "month"])
    .csv("CRAFT/" + year0 + "/flightplans", compression="gzip", mode="overwrite")
)